# Пример: dense GAN и RGB DCGAN для лиц на PyTorch и Intel Arc

Этот ноутбук повторяет корректное сравнение dense GAN и DCGAN на LFW,
но использует PyTorch. Если `torch.xpu.is_available()` возвращает
истину, вычисления выполняются на Intel Arc; иначе автоматически
выбирается CPU. Выбор устройства выводится в первой ячейке и не
скрывается от студента.

Метки персон LFW не участвуют в состязательной функции потерь. GAN
обучается на изображениях лиц без классовых целей.

## История одного мини-пакета

Художник получает $B$ наборов из 100 случайных чисел и рисует $B$ лиц.
Проверяющий видит эти рисунки и настоящие фотографии, затем выдаёт по
одному числу на картинку. В первой половине шага исправляются веса
проверяющего, во второй — веса художника. PyTorch явно показывает эти
действия через `zero_grad()`, `backward()` и `step()`.

Указанные под фотографиями ID нужны только человеку, который смотрит
на данные. Ни генератор, ни дискриминатор их не получают.

### Словарь перед первой реализацией

Параметры — числа в слоях, которые можно менять. Прямой проход —
создание картинки и её проверка. Потеря — одно число, показывающее
несоответствие текущего ответа цели. Градиент — подсказка, как слегка
сдвинуть параметры для уменьшения ошибки. В PyTorch `backward()`
находит градиенты, а `step()` применяет поправку оптимизатора.

## Этап 1. Устройство, профиль и соглашения о формах

**Действие:** выбрать XPU при его доступности, иначе CPU; затем выбрать
соответствующий вычислительный профиль. Профиль Arc использует
8 000 лиц и 120 эпох, CPU — 2 500 лиц и 80 эпох.

**Проверка:** напечатанный `device` должен быть `xpu` при успешной
настройке Intel Arc. Тензоры и параметры моделей далее переносятся на
это же устройство.

`(B,3,64,64)` читается слева направо: B лиц, 3 цветовых канала, 64
строки и 64 столбца. Это соглашение PyTorch для свёрток. Случайный шум
имеет форму `(B,100)`, а ответ проверяющего `(B,1)`: одно сырое число
на изображение.

In [ ]:
from __future__ import annotations

# Path и time обслуживают артефакты/измерение длительности, но не
# входят в вычислительный граф PyTorch.
from pathlib import Path
import time

# Явно выбираем встроенный Jupyter-backend, чтобы plt.show() не
# открывал внешнее окно Figure при запуске ячеек.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')
# NumPy хранит выборку и историю, torch — тензоры, параметры и
# автоматическое дифференцирование; nn содержит строительные блоки.
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import fetch_lfw_people
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Один seed фиксирует веса, шум и воспроизводимое разбиение LFW.
SEED = 42
IMAGE_SIZE, CHANNELS, NOISE_DIM = 64, 3, 100
REAL_LABEL, SNAPSHOT_EVERY = 0.9, 10
# XPU выбирается только при фактической доступности; этот же код
# безопасно продолжит работу на CPU, если Intel Arc не настроен.
device = torch.device('xpu' if torch.xpu.is_available() else 'cpu')
PROFILE = 'arc' if device.type == 'xpu' else 'cpu'
PROFILES = {
    'cpu': {'max_examples': 2_500, 'epochs': 80, 'batch_size': 32},
    'arc': {'max_examples': 8_000, 'epochs': 120, 'batch_size': 128},
}
# Берём связанные лимит/эпохи/batch из одного профиля, а не меняем
# отдельные числа в цикле и не получаем случайно смешанный запуск.
config = PROFILES[PROFILE]
DATA_CACHE = Path('artifacts/scikit_learn')
ARTIFACT_DIR = Path('artifacts/lfw_face_gan_pytorch') / PROFILE
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Seed задаётся до конструкторов моделей и первых torch.randn.
torch.manual_seed(SEED)
if device.type == 'xpu':
    torch.xpu.manual_seed_all(SEED)
print(f'PyTorch {torch.__version__}; устройство: {device}; профиль: {PROFILE}')
print('Ожидаемые формы: z=(B,100), x=(B,3,64,64), D(x)=(B,1).')

## Этап 2. LFW и переход к пакетной семантике PyTorch

**Вход:** цветные изображения LFW в форме `(N,H,W,C)`.
**Действие:** изменить размер, нормировать, поменять порядок осей на
PyTorch-формат `(N,C,H,W)` и создать `DataLoader`.
**Выход:** каждый обучающий пакет имеет форму `(B,3,64,64)`.

Сначала LFW приходит в привычном для изображений порядке
`(N,H,W,C)`. `permute` переставляет только оси и делает
`(N,C,H,W)`, не меняя сами цвета. Затем значения `[0,1]` переводятся
в `[-1,1]`, чтобы настоящие лица и выход `tanh` генератора были в
одинаковой шкале. В `DataLoader` кладутся только изображения: ID людей
не нужны ни для одной ошибки GAN.

In [ ]:
# LFW даёт массив NHWC и ID персон. ID не будут переданы в loader/loss:
# GAN моделирует распределение лиц, а не распознаёт человека.
lfw = fetch_lfw_people(data_home=str(DATA_CACHE), color=True, resize=0.5, min_faces_per_person=0)
selected = np.random.default_rng(SEED).permutation(len(lfw.images))[:config['max_examples']]
# Значения уже float32 в [0,1]; повторное деление на 255/127.5 здесь
# сдвинуло бы изображения почти к -1 и визуализация стала бы чёрной.
images_unit = torch.from_numpy(lfw.images[selected].astype('float32'))
assert 0.0 <= float(images_unit.min()) <= float(images_unit.max()) <= 1.0
diagnostic_unit = images_unit[:min(256, len(images_unit))]
print('LFW до нормализации (первые 256 лиц):', torch.quantile(
    diagnostic_unit, torch.tensor([0.00, 0.01, 0.50, 0.99, 1.00])
).tolist())
# Scikit-learn/Matplotlib используют NHWC, а Conv2d ждёт NCHW.
# permute не изменяет значения, а переставляет оси N,H,W,C→N,C,H,W.
images = images_unit.permute(0, 3, 1, 2)
images = torch.nn.functional.interpolate(images, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)
# Теперь диапазон реальных изображений совпадает с tanh-выходом G.
images = 2.0 * images - 1.0
person_ids = lfw.target[selected]
assert tuple(images.shape) == (config['max_examples'], CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
assert torch.isfinite(images).all() and float(images.min()) >= -1.0 and float(images.max()) <= 1.0
diagnostic_real = images[:min(256, len(images))]
print('LFW после нормализации (первые 256 лиц):', torch.quantile(
    diagnostic_real, torch.tensor([0.00, 0.01, 0.50, 0.99, 1.00])
).tolist())
# TensorDataset намеренно содержит лишь images. Loader отдаёт кортеж
# из одного тензора (B,3,64,64); drop_last сохраняет одинаковый B.
loader = DataLoader(TensorDataset(images), batch_size=config['batch_size'], shuffle=True, drop_last=True)
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for axis, image, person_id in zip(axes.flat, images[:12], person_ids[:12]):
    axis.imshow(np.clip(((image.permute(1, 2, 0) + 1.0) / 2.0).numpy(), 0.0, 1.0), vmin=0.0, vmax=1.0)
    axis.set_title(f'id={int(person_id)}', fontsize=8)
    axis.axis('off')
fig.suptitle('LFW: идентификаторы показаны только для диагностики')
plt.tight_layout(); plt.show(); plt.close(fig)
print('Первый пакет:', tuple(next(iter(loader))[0].shape), '; метки персон в DataLoader не передаются.')

## Этап 3. Dense GAN и DCGAN

`DenseGenerator` должен развернуть вектор признаков в RGB-карту только
в конце. `DCGenerator` строит пространственную карту постепенно через
транспонированные свёртки. У обоих дискриминаторов последний слой
возвращает логит, поэтому используется `BCEWithLogitsLoss`.

**Dense GAN** делает длинную строку из всех 12 288 RGB-значений и не
знает соседства пикселей заранее. **DCGAN** сохраняет карту: начинает
с маленькой внутренней карты и увеличивает её до лица. Одинаковые
свёрточные правила работают в разных местах изображения, поэтому
DCGAN лучше приспособлен к глазам, контурам и локальным деталям.

Логит — ещё не вероятность, а свободная оценка: он может быть любым
числом. `BCEWithLogitsLoss` сама устойчиво соединяет sigmoid и
бинарную кросс-энтропию, поэтому отдельный sigmoid добавлять нельзя.

In [ ]:
class DenseGenerator(nn.Module):
    # G принимает z=(B,100). Dense-сеть смешивает все координаты z,
    # затем последний слой создаёт B×(64·64·3) чисел, а view вернёт
    # пространственную форму без изменения количества элементов.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(NOISE_DIM, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 2048),
            nn.LeakyReLU(0.2),
            nn.Linear(2048, IMAGE_SIZE * IMAGE_SIZE * CHANNELS),
            nn.Tanh(),
        )

    def forward(self, z):
        # pixels: (B, 12288), значения ограничены Tanh интервалом [-1,1].
        pixels = self.net(z)
        # Conv2d/ConvTranspose2d далее используют порядок (B,C,H,W).
        return pixels.view(-1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

class DenseDiscriminator(nn.Module):
    # D «забывает» соседство пикселей через Flatten и возвращает один
    # логит (B,1). Sigmoid отсутствует: BCEWithLogitsLoss объединяет
    # sigmoid и BCE устойчивее, чем две отдельные операции.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMAGE_SIZE * IMAGE_SIZE * CHANNELS, 2048),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(2048, 1024),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(1024, 1),
        )

    def forward(self, x):
        # x=(B,3,64,64), ответ net=(B,1); это не вероятность до loss.
        return self.net(x)

class DCGenerator(nn.Module):
    # DCGAN подаёт z как карту (B,100,1,1). Пять транспонированных
    # свёрток создают размеры 1→4→8→16→32→64; общие ядра формируют
    # пространственно согласованные детали, а не независимые пиксели.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(NOISE_DIM, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, CHANNELS, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        # None добавляет две оси длины 1: z остаётся тем же набором
        # чисел, но получает форму, ожидаемую ConvTranspose2d.
        seed_map = z[:, :, None, None]
        return self.net(seed_map)

class DCDiscriminator(nn.Module):
    # Четыре stride=2 Conv2d сжимают 64→32→16→8→4. Dropout активен
    # только в train-режиме фазы D; итоговый Linear выдаёт логит (B,1).
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(CHANNELS, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Flatten(),
            nn.Linear(4 * 4 * 512, 1),
        )

    def forward(self, x):
        return self.net(x)

# Сопоставление хранит классы, а не созданные сети: для каждого
# варианта ниже будут инициализированы независимые параметры.
factories = {'dense': (DenseGenerator, DenseDiscriminator), 'dcgan': (DCGenerator, DCDiscriminator)}
for name, (G, D) in factories.items():
    generator = G().to(device)
    discriminator = D().to(device)
    # z на том же device исключает неявное копирование CPU↔XPU.
    z = torch.randn(config['batch_size'], NOISE_DIM, device=device)
    assert tuple(generator(z).shape) == (config['batch_size'], CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
    assert tuple(discriminator(generator(z)).shape) == (config['batch_size'], 1)
    print(f'{name}: G={sum(p.numel() for p in generator.parameters()):,}, D={sum(p.numel() for p in discriminator.parameters()):,}')

## Этап 4. Два независимых обратных прохода

Во время обучения дискриминатора синтетические лица строятся в
`torch.no_grad()`, а генератор временно переводится в `eval()`: его
параметры и статистики BatchNorm не меняются. Во время обучения
генератора дискриминатор переводится в `eval()` и его параметры
временно получают `requires_grad=False`, но операции остаются в
вычислительном графе и передают производную к сгенерированному лицу.

### Что означают похожие команды PyTorch

`generator.eval()` и `discriminator.eval()` выбирают режим слоёв:
выключают Dropout и запрещают BatchNorm обновлять накопленные
статистики. Это не выключает вычисление градиента само по себе.
`torch.no_grad()` действительно не строит граф производных — его
применяют, когда D учится на готовом рисунке G. `requires_grad_(False)`
замораживает веса D, но оставляет путь через его операции; это нужно,
когда G получает замечание от D.

В фазе D `zero_grad()` убирает следы прошлого пакета, `backward()`
записывает замечания в веса D, а `opt_d.step()` их меняет. В фазе G
те же три действия относятся к G. Это не один общий шаг, а два разных
урока с разными тетрадями параметров.

D возвращает логит $\ell$ — произвольное число до вероятности. Если
мысленно применить sigmoid, получится
$p=\sigma(\ell)=1/(1+e^{-\ell})$. Для цели $y$ ошибка одной картинки
равна

$$
\operatorname{BCE}(y,p)=-y\log p-(1-y)\log(1-p).
$$

`BCEWithLogitsLoss` получает именно логит $\ell$ и объединяет sigmoid
с этой формулой внутри устойчивого вычисления. Поэтому отдельный
`sigmoid` перед функцией потерь здесь не нужен.

### От математической формулы к PyTorch: что делает каждый объект

В одном шаге существуют три разных вида объектов. `real_images`,
`discriminator_noise`, `fake_images` и `generated` — временные тензоры
данных. Параметры слоёв G и D — тензоры-листья `nn.Parameter`; именно
у них `requires_grad=True` и именно для них PyTorch может накопить
производную. `d_loss` и `g_loss` — скаляры: одно число ошибки, с
которого `backward()` начинает движение по графу в обратную сторону.

### Инфографика: две фазы одного пакета

![Два независимых обновления GAN: обучение дискриминатора, затем обучение генератора](assets/diagrams/gan_two_phase_training.svg)

В левой фазе `torch.no_grad()` означает, что рисунок G(z_D) нужен D
только как готовый пример: путь к параметрам G не записывается. В
правой фазе `requires_grad_(False)` относится лишь к весам D. Операции
D остаются в графе, так что замечание от `g_loss` может пройти через
них к изображению `generated` и далее к весам G.

### Внутри Autograd: динамический граф и поля `.grad`

![Жизненный цикл динамического графа PyTorch и обновления Adam](assets/diagrams/pytorch_autograd.svg)

При обычном прямом проходе PyTorch создаёт динамический ориентированный
граф: результат каждой дифференцируемой операции хранит ссылку
`grad_fn` на правило локальной производной. Это не отдельный объект,
который студент должен собрать руками. `loss.backward()` начинает со
скаляра loss, применяет цепное правило к этим локальным правилам и
складывает итоговую производную в поле `.grad` каждого достижимого
параметра-листа с `requires_grad=True`.

`zero_grad(set_to_none=True)` очищает старые `.grad`, потому что
PyTorch по умолчанию **суммирует** производные нескольких `backward()`.
`optimizer.step()` не вычисляет градиенты: он только читает поля `.grad`
параметров, переданных при создании именно этого оптимизатора, и
применяет Adam. После обычного `backward()` временный граф освобождается,
так как для одного шага он больше не нужен.

Обозначим все веса художника через $\theta_G$, а все веса проверяющего
через $\theta_D$. Потеря — одно число, а её производная по одному весу
показывает, что произойдёт при его очень малом сдвиге. Поэтому идея
обучения записывается так:

$$
\theta\leftarrow\theta-\eta\nabla_\theta\mathcal L.
$$

В первой фазе вычисляется

$$
\mathcal L_D=\tfrac12\left[
\operatorname{BCE}(0.9,D_{\theta_D}(\mathbf x_{\rm real}))+
\operatorname{BCE}(0,D_{\theta_D}(G_{\theta_G}(\mathbf z_D)))
\right],
$$

и изменяется только $\theta_D$. Во второй фазе вычисляется

$$
\mathcal L_G=\operatorname{BCE}
\left(1,D_{\theta_D}(G_{\theta_G}(\mathbf z_G))\right),
$$

и изменяется только $\theta_G$. Шум $\mathbf z$ — входная карточка,
а не параметр: он не сохраняется как «правильный ответ» и каждый раз
выбирается заново.

PyTorch строит граф прямого прохода автоматически, пока нужные
тензоры имеют `requires_grad=True`. `zero_grad()` очищает старые поля
`parameter.grad`, потому что PyTorch по умолчанию складывает новые
производные со старыми. `loss.backward()` применяет цепное правило и
помещает градиент той же формы, что и вес, в `parameter.grad`.
`optimizer.step()` читает эти градиенты и делает шаг Adam.

Во время фазы G путь производной остаётся

$$
\frac{\partial\mathcal L_G}{\partial\theta_G}
=
\frac{\partial\mathcal L_G}{\partial D}
\frac{\partial D}{\partial G}
\frac{\partial G}{\partial\theta_G}.
$$

`requires_grad_(False)` запрещает складывать градиент в веса D, но
операции D всё ещё нужны для среднего множителя в этой цепочке.
`torch.no_grad()` сильнее: он вообще не записывает операции и потому
годится только для синтеза подделок в фазе D или для визуализации.

### Один мини-пакет: математические объекты и PyTorch

В таблице указаны реальные имена из тела `train_gan`. Временные тензоры
исчезают после шага; оптимизаторы хранят состояние только для весов G
и D, переданных им при создании.

| Математический объект | Имя в коде | Форма и роль | Градиентный статус |
| --- | --- | --- | --- |
| Реальные лица $\mathbf x_{\rm real}$ | `real_images` | `(B,3,64,64)` | Вход D в фазе D. |
| Шум D $\mathbf z_D$ | `discriminator_noise` | `(B,100)` | В `no_grad`, не строит путь к G. |
| Подделки D | `fake_images` | `(B,3,64,64)` | Готовые данные для D, без графа G. |
| Логиты D | `real_logits`, `fake_logits` | `(B,1)` | Участвуют в `real_loss`, `fake_loss`. |
| Потеря D | `d_loss` | скаляр $\mathcal L_D$ | После `backward()` даёт `.grad` параметрам D. |
| Шум G $\mathbf z_G$ | `generator_noise` | `(B,100)` | Новый временный вход G. |
| Выход G | `generated` | `(B,3,64,64)` | Остаётся в графе обратного прохода. |
| Логит D для G | `fake_logits_for_generator` | `(B,1)` | Сравнивается с целью 1. |
| Потеря G | `g_loss` | скаляр $\mathcal L_G$ | После `backward()` даёт `.grad` параметрам G. |

**Как читать форму и статус.** `generated` имеет форму
`(B,3,64,64)` и содержит обычные значения пикселей; это не набор
параметров. Но у него есть связь `grad_fn` с операциями G, поэтому
результат D может передать обратно подсказку для каждого пикселя.
`generator_noise` имеет форму `(B,100)`, однако его значения не
оптимизируются: новая случайная карточка лишь выбирает точку, в которой
общая функция G должна быть улучшена.

Для логитного выхода D та же цепочка в строгой векторной записи имеет
вид

$$
\nabla_{\theta_G}\mathcal L_G
=J_{G,\theta_G}^{\mathsf T}
J_{D,\mathbf x}^{\mathsf T}
\nabla_{\ell}\mathcal L_G.
$$

$J_{G,\theta_G}$ показывает влияние всех параметров G на пиксели,
$J_{D,\mathbf x}$ — влияние пикселей на логит D. Autograd не строит
эти огромные таблицы целиком: `backward()` сразу передаёт их действие
на вектор ошибки от правой части формулы к левой. Это обратное
распространение — последовательность малых локальных применений
цепного правила, а не буквальное умножение гигантских матриц.

### API как точная последовательность состояний

1. `opt_d.zero_grad(set_to_none=True)` очищает прежние поля
   `parameter.grad` у параметров D. Иначе PyTorch сложил бы новый
   градиент с тем, который остался от предыдущего пакета.
2. Обычный прямой проход создаёт `real_logits`, `fake_logits` и
   единственный скаляр `d_loss`.
3. `d_loss.backward()` идёт по записанному графу назад и кладёт в
   каждый параметр D тензор `parameter.grad` той же формы.
4. `opt_d.step()` читает именно эти поля и меняет только веса D.
5. Затем параметры D получают `requires_grad_(False)`, очищается
   градиент G и строится новый путь до `g_loss`.
6. `g_loss.backward()` проходит через операции D к `generated`, но не
   создаёт новые `.grad` у замороженных весов D; `opt_g.step()` меняет
   только веса G.

`eval()` и `train()` не входят в эти шесть математических действий:
они выбирают поведение Dropout и BatchNorm. `no_grad()` запрещает
создать граф совсем, а `requires_grad_(False)` запрещает накапливать
градиент лишь у выбранных листовых параметров. Это три разных
механизма, хотя в коде они стоят рядом.

In [ ]:
# D возвращает логиты (B,1). BCEWithLogitsLoss применяет sigmoid
# внутри устойчивой формулы и ожидает вещественные цели той же формы.
criterion = nn.BCEWithLogitsLoss()
# Fixed noise нужен исключительно для сравнимых визуальных снимков.
fixed_noise = torch.randn(64, NOISE_DIM, device=device)

def save_grid(generator, path):
    # eval() меняет поведение модулей: BatchNorm читает сохранённые
    # статистики, Dropout отключается. Это не отменяет requires_grad.
    generator.eval()
    # Здесь no_grad уместен: визуализация не должна строить граф и
    # создавать .grad, так как последующего backward() не будет.
    with torch.no_grad():
        images = generator(fixed_noise).cpu()
    # Выход G: (64,3,64,64), [-1,1]. Matplotlib ждёт RGB в [0,1]
    # и порядок H,W,C, поэтому одновременно инвертируем шкалу и оси.
    images = ((images + 1) / 2).permute(0, 2, 3, 1).numpy()
    # Возвращаем генератор в train до следующего обучающего пакета.
    generator.train()
    fig, axes = plt.subplots(8, 8, figsize=(8, 8))
    for axis, image in zip(axes.flat, images):
        axis.imshow(np.clip(image, 0.0, 1.0), vmin=0.0, vmax=1.0)
        axis.axis('off')
    plt.tight_layout(pad=0.05)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def train_gan(name, factory):
    # Свежие G и D исключают перенос весов/моментов между dense и DCGAN.
    generator = factory[0]().to(device)
    discriminator = factory[1]().to(device)
    # Adam хранит моменты по параметрам, поэтому оптимизаторы строго
    # раздельны: opt_g знает θ_G, opt_d — θ_D.
    opt_g = torch.optim.Adam(generator.parameters(), 2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(discriminator.parameters(), 2e-4, betas=(0.5, 0.999))
    history = {'generator_loss': [], 'discriminator_loss': []}
    started = time.perf_counter()
    for epoch in range(1, config['epochs'] + 1):
        d_values = []
        g_values = []
        for (real_images,) in loader:
            # Loader отдаёт единственный NCHW-тензор (B,3,64,64).
            # .to(device) перемещает данные на тот же CPU/XPU, где θ.
            real_images = real_images.to(device)
            batch_size = len(real_images)

            # === Фаза 1: оптимизируется θ_D ===
            # PyTorch суммирует градиенты в .grad по умолчанию.
            # zero_grad очищает старые ∂L_D/∂θ_D до нового backward.
            opt_d.zero_grad(set_to_none=True)
            # G временно в eval: у BatchNorm G не изменяются running
            # statistics. Это о режиме слоя, не о заморозке весов.
            generator.eval()
            with torch.no_grad():
                # z_D=(B,100); no_grad не записывает G(z_D) в граф.
                # Таким образом L_D не может породить градиенты θ_G.
                discriminator_noise = torch.randn(batch_size, NOISE_DIM, device=device)
                fake_images = generator(discriminator_noise)
            # В дальнейшем G обучается, поэтому возвращаем train-состояние.
            generator.train()

            # real/fake_targets имеют форму (B,1), как логиты D.
            # 0.9 — сглаженная цель реальных, 0 — синтетических лиц.
            real_targets = torch.full((batch_size, 1), REAL_LABEL, device=device)
            fake_targets = torch.zeros(batch_size, 1, device=device)

            # D вызывается в train-режиме: Dropout создаёт регуляризацию.
            # Каждый BCE — скаляр среднего по B; их среднее балансирует
            # два источника данных в итоговой скалярной d_loss.
            real_logits = discriminator(real_images)
            fake_logits = discriminator(fake_images)
            real_loss = criterion(real_logits, real_targets)
            fake_loss = criterion(fake_logits, fake_targets)
            d_loss = 0.5 * (real_loss + fake_loss)

            # backward обходит граф d_loss→D и записывает ∂L_D/∂θ_D
            # в параметрах D. У G графа здесь нет из-за no_grad.
            d_loss.backward()
            # Только opt_d читает эти .grad и меняет θ_D.
            opt_d.step()

            # === Фаза 2: оптимизируется θ_G ===
            # Заморозка параметров D экономит память и исключает
            # накопление ∂L_G/∂θ_D. Это НЕ no_grad: операции D должны
            # остаться в графе, чтобы передать ∂L_G/∂G(z) генератору.
            for parameter in discriminator.parameters():
                parameter.requires_grad_(False)
            # D.eval выключает его Dropout: G получает стабильный
            # учебный сигнал. eval и requires_grad решают разные задачи.
            discriminator.eval()
            # Очистка относится только к θ_G; градиенты D в этой фазе
            # не требуются, потому что requires_grad=False.
            opt_g.zero_grad(set_to_none=True)

            # z_G независимо от z_D. generated — (B,3,64,64) и на
            # этот раз остаётся в графе, поскольку G будет обновляться.
            generator_noise = torch.randn(batch_size, NOISE_DIM, device=device)
            generated = generator(generator_noise)
            generator_targets = torch.ones(batch_size, 1, device=device)

            # fake_logits_for_generator=(B,1). Цель 1 — требование
            # повысить правдоподобие «реального» у D для G(z_G).
            fake_logits_for_generator = discriminator(generated)
            g_loss = criterion(fake_logits_for_generator, generator_targets)

            # backward проводит сигнал L_G→D-операции→generated→θ_G.
            # Параметры D заморожены, поэтому .grad появится только у G.
            g_loss.backward()
            # Только opt_g применяет ∂L_G/∂θ_G и обновляет генератор.
            opt_g.step()

            # Восстанавливаем D для первой фазы следующего пакета.
            discriminator.train()
            for parameter in discriminator.parameters():
                parameter.requires_grad_(True)

            # item извлекает скаляр для протокола. После вызова он не
            # участвует в градиентах или оптимизации.
            d_values.append(d_loss.item())
            g_values.append(g_loss.item())
        # Храним среднее по эпохе, а не случайный последний пакет.
        history['discriminator_loss'].append(float(np.mean(d_values)))
        history['generator_loss'].append(float(np.mean(g_values)))
        if epoch == 1 or epoch % SNAPSHOT_EVERY == 0 or epoch == config['epochs']:
            save_grid(generator, ARTIFACT_DIR / f'{name}_epoch_{epoch:03d}.png')
        print(f'{name}, эпоха {epoch:03d}: D={history["discriminator_loss"][-1]:.4f}, G={history["generator_loss"][-1]:.4f}')
    return generator, discriminator, history, time.perf_counter() - started

## Этап 5. Сопоставление результатов

Обе архитектуры обучаются в одинаковом порядке на том же срезе LFW.
Сопоставьте не только финальные значения состязательных потерь, но и
сетки лиц, скорость, число параметров и признаки коллапса мод.

Фиксированные 64 шумовые карточки показывают, как меняется один и тот
же набор будущих лиц. Перед показом PyTorch-порядок `(C,H,W)` меняется
на `(H,W,C)`, а диапазон `[-1,1]` — на `[0,1]`: это требование
Matplotlib, не изменение данных модели.

Обе линии потерь нужно читать как дневник игры. Низкая ошибка одной
стороны иногда лишь означает, что сейчас она сильнее соперника.
Содержательный вывод делают по сеткам, разнообразию лиц, конечности
потерь и признакам коллапса, а не по одной последней цифре.

In [ ]:
results = {name: train_gan(name, factory) for name, factory in factories.items()}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, _, history, seconds) in results.items():
    axes[0].plot(history['discriminator_loss'], label=f'D, {name}')
    axes[1].plot(history['generator_loss'], label=f'G, {name}; {seconds / 60:.1f} мин')
for axis, title in zip(axes, ('Потеря дискриминатора', 'Потеря генератора')):
    axis.set(title=title, xlabel='Эпоха', ylabel='BCE по логитам'); axis.grid(alpha=0.3); axis.legend()
plt.tight_layout(); plt.show()
for name, (_, _, history, seconds) in results.items():
    assert np.isfinite(history['generator_loss']).all() and np.isfinite(history['discriminator_loss']).all()
    print(f'{name}: {seconds / 60:.1f} мин, D={history["discriminator_loss"][-1]:.4f}, G={history["generator_loss"][-1]:.4f}')